# 09: ablacje: czy potrzebna jest kamera B? czy potrzebna jest geometria?

Ten sam zbiór `rs_video_x4k` i **ta sama zasada co w głównym porównaniu architektur: każdy model uczony od zera ma 200 epok**
(bs 16, lr 3e-4, OneCycle). Pary porównywane ze sobą różnią się więc tylko jedną rzeczą. Dwa pytania:

**A. Kamera B.** Te same modele co w 08, ale **zamiast obrazu z kamery B sieć dostaje kopię obrazu A** (`--single`).
Architektura i liczba wag bez zmian, sieć nie ma tylko drugiej chwili odczytu. Na etapie 1 (MotionNet, sam ruch kamery):
+8,9 dB z kamerą B, +0,2 dB bez niej (`docs/etap1_motionnet.md`); tu sprawdzamy to na ruchu obiektów.

**B. Geometria („korelacja + geometria”).** Sieć z `corr` (ta sama architektura, 6,2 M wag, uczona od zera) zamiast obrazu
przewiduje **przesunięcie A→B**, a stała geometria RS v1 zamienia je na obraz, dokładnie jak w RAFT + geometria v1.
To szczebel między `corr` (dopasowanie bez geometrii) a RAFT + geometrią (wstępnie wytrenowane dopasowanie + geometria):
* dużo lepiej niż `corr` → kluczem jest geometria,
* blisko `corr` → kluczem jest dobre dopasowanie RAFT (wstępny trening na zbiorach przepływu).

| sekcja | run | `--mode` | para do porównania | czas (L4) |
|---|---|---|---|---|
| 3 | zwykły U-Net bez kamery B | `plain --single` | `*_unet_plain_long` z 08 (200 ep., 30,94 dB) | ~70 min |
| 4 | `corr` z kamerą B, 200 epok | `corr` | para dla sekcji 5 i 6 (`corr` z 08 miał tylko 80 epok) | ~2 h |
| 5 | `corr` bez kamery B | `corr --single` | `*_corr200` (sekcja 4) | ~2 h |
| 6 | korelacja + geometria | `corrgeom` | `*_corr200` i RAFT + geometria v1 z 04 (32,86 bez uczenia / 34,30 dB) | ~2–2,5 h |

`psnr` liczony jak w 04 (obszar widoczny w A); `psnr_full` = cały obraz (wspólna miara dla wszystkich modeli).
Sekcja 7 (opcjonalna): testy kontrolne współczynnika uczenia i rozmiaru sieci, też po 200 epok.
**Sekcje 3–7 są niezależne:** do równoległych treningów zrób kopie tego pliku (jeden plik = jedno jądro Colaba) i w każdej
uruchom SETUP, 1, 2, potem swoją sekcję. Jeśli `*_corr200` jest już uruchomiony w kopii notatnika 08, pomiń sekcję 4.

In [ ]:
# === SETUP: uruchamiaj zawsze jako pierwszą komórkę ===
# Kod jest pobierany z GitHuba na maszynę Colaba (lokalne pliki z VS Code tam nie trafiają).
# Po zmianie kodu: commit + push w VS Code, potem ponownie ta komórka (robi git pull).
REPO = 'kordiusz/RollingShutter'   # repozytorium na GitHubie
BRANCH = 'main'

import os, sys, subprocess, getpass, importlib
from google.colab import drive
drive.mount('/content/drive')

TOKEN_FILE = '/content/drive/MyDrive/rs_project/.github_token'   # opcjonalnie: token zapisany na Drive
CODE = '/content/code'

def _git(*args):
    r = subprocess.run(['git', *args], capture_output=True, text=True)
    if r.returncode != 0:
        msg = (r.stderr or r.stdout).replace(token, '***') if 'token' in globals() else r.stderr
        raise RuntimeError('git ' + args[0] + ' nie powiodło się:\n' + msg)
    return r.stdout

if not os.path.exists(CODE):
    token = (open(TOKEN_FILE).read().strip() if os.path.exists(TOKEN_FILE)
             else getpass.getpass('GitHub token (fine-grained, tylko odczyt): '))
    _git('clone', '-q', '-b', BRANCH, f'https://{token}@github.com/{REPO}.git', CODE)
else:
    _git('-C', CODE, 'pull', '-q')
os.chdir(CODE)
sys.path.insert(0, f'{CODE}/src')
import project_paths as P
importlib.reload(P)
P.ensure_dirs()
print(_git('log', '-1', '--format=kod: %h  %s  (%cd)', '--date=short'))
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 1. Zbiór danych (rozpakowanie z Drive na dysk maszyny Colaba)

In [ ]:
import os, glob, shutil, subprocess
DATASET_NAME = 'rs_video_x4k'
z = P.find_dataset_zip(DATASET_NAME)
assert z, f'Brak {DATASET_NAME}.zip w {P.DATASETS} - wrzuć tam spakowany folder data/{DATASET_NAME}'
UNZ = f'/content/unz_{DATASET_NAME}'
DONE = f'{UNZ}/.rozpakowane'          # znacznik: rozpakowanie skończyło się bez błędu
if not os.path.exists(DONE):
    shutil.rmtree(UNZ, ignore_errors=True)            # resztki po przerwanym rozpakowaniu
    LOCAL_ZIP = f'/content/{DATASET_NAME}.zip'
    # najpierw kopia na dysk Colaba: unzip czytający wprost z Drive potrafi się urwać ("zipfile read error")
    print(f'kopiuję {z} ({os.path.getsize(z):,} B) na dysk Colaba ...')
    subprocess.run(['cp', str(z), LOCAL_ZIP], check=True)
    t = subprocess.run(['unzip', '-tq', LOCAL_ZIP], capture_output=True, text=True)
    assert t.returncode == 0, ('Archiwum uszkodzone - wgraj zip na Drive ponownie (lokalnie ma 1 233 434 992 B):\n'
                               + (t.stdout + t.stderr)[-1500:])
    print('archiwum OK, rozpakowuję ...')
    subprocess.run(['unzip', '-q', LOCAL_ZIP, '-d', UNZ], check=True)
    os.remove(LOCAL_ZIP)
    open(DONE, 'w').close()
# folder z podfolderami train/ i val/ (niezależnie od tego, jak spakowano)
DATA = next(os.path.dirname(p) for p in sorted(glob.glob(f'{UNZ}/**/train', recursive=True)))
assert os.path.isdir(f'{DATA}/val'), f'brak {DATA}/val - usuń {UNZ} i uruchom komórkę ponownie'
print('DATA =', DATA, '|', len(os.listdir(f'{DATA}/train')), 'train /', len(os.listdir(f'{DATA}/val')), 'val')
WORKERS = min(8, os.cpu_count())

## 2. Nazwy runów
Przy wznawianiu innego dnia wpisz datę istniejących folderów z `runs/`.

In [ ]:
from datetime import date
from IPython.display import Image, display
DAY = str(date.today())          # np. '2026-11-12' przy wznawianiu
EPOCHS, BS, LR = 200, 16, 3e-4   # ta sama zasada dla wszystkich modeli uczonych od zera
JOBS = {   # nazwa: (tryb, bez kamery B?)
    'plain_single': ('plain', True),
    'corr200':      ('corr', False),
    'corr_single':  ('corr', True),
    'corrgeom':     ('corrgeom', False),
}
RUNS = {'plain_single': f'{DAY}_unet_plain_single', 'corr200': f'{DAY}_corr200',
        'corr_single': f'{DAY}_corr_single', 'corrgeom': f'{DAY}_corrgeom'}

def _check_setup():
    """IPython leaves a !command unexpanded when any {name} in it is undefined, so check first."""
    missing = [v for v in ('P', 'DATA', 'WORKERS') if v not in globals()]
    if missing:
        raise RuntimeError(f'brak {missing}: najpierw uruchom SETUP i komórkę 1 (zbiór danych) w tej sesji '
                           f'i poczekaj, aż wypisze "DATA = ... | 2397 train / 321 val"')

def train(job, resume=False):
    _check_setup()
    mode, single = JOBS[job]
    run = RUNS[job]
    flags = ' --single' if single else ''
    res = f' --resume "{P.RUNS / run / "last.pt"}"' if resume else ''
    cmd = (f'python src/train_unet.py --data "{DATA}" --mode {mode}{flags} --epochs {EPOCHS} --bs {BS} --lr {LR} '
           f'--workers {WORKERS} --out "{P.RUNS}" --name {run}{res}')
    print(cmd)
    get_ipython().system(cmd)
    prev = sorted((P.RUNS / run).glob('preview_ep*.png'))
    if prev:
        print(prev[-1]); display(Image(str(prev[-1])))
print(RUNS)

## 3. A: zwykły U-Net bez kamery B, 200 epok (~70 min na L4)
Po rozłączeniu: SETUP, 1, 2, potem `train('plain_single', resume=True)`.

In [ ]:
train('plain_single')

## 4. `corr` z kamerą B, 200 epok (~2 h)
Para dla sekcji 5 i 6. Pomiń, jeśli `*_corr200` jest już uruchomiony w kopii notatnika 08.

In [ ]:
train('corr200')

## 5. A: `corr` bez kamery B, 200 epok (~2 h)

In [ ]:
train('corr_single')

## 6. B: korelacja + geometria, 200 epok (~2–2,5 h)
Podgląd: RS A | RS B | wynik | błąd ×4 | wzorzec.

In [ ]:
train('corrgeom')

## 7. (opcjonalnie) Testy kontrolne: współczynnik uczenia i rozmiar sieci

Nie strojenie, tylko odpowiedź na dwa pytania, które łatwo zadać przy obronie. Puszczaj po sekcjach 3–6, jeśli zostanie czas.
Też 200 epok, więc pary są z głównego porównania.

| test | run | pytanie | para | czas (L4) |
|---|---|---|---|---|
| 7a | zwykły U-Net, `--lr 1e-4` i `--lr 1e-3` | czy U-Net miał zły współczynnik uczenia? | `*_unet_plain_long` (lr 3e-4) | 2 × ~70 min |
| 7b | `corr --width 64` (~4× więcej wag, 24,4 M) | czy sieć bez geometrii jest po prostu za mała? | `*_corr200` (width 32, 6,2 M) | ~4–5 h (szacunek) |

Jak czytać: jeśli 7a mieści się w ok. 0,5 dB, wynik nie zależy od lr; jeśli 7b też stoi w okolicy 31 dB, granica wynika
z architektury (braku geometrii), a nie z liczby wag. Ważniejszy jest test 7b.

In [ ]:
CONTROL = {   # nazwa runu: (tryb, lr, szerokość)
    f'{DAY}_unet_plain_lr1e-4': ('plain', 1e-4, 32),
    f'{DAY}_unet_plain_lr1e-3': ('plain', 1e-3, 32),
    f'{DAY}_corr_w64':          ('corr', 3e-4, 64),
}

def train_control(run, resume=False):
    _check_setup()
    mode, lr, width = CONTROL[run]
    res = f' --resume "{P.RUNS / run / "last.pt"}"' if resume else ''
    cmd = (f'python src/train_unet.py --data "{DATA}" --mode {mode} --width {width} --epochs {EPOCHS} --bs {BS} '
           f'--lr {lr} --workers {WORKERS} --out "{P.RUNS}" --name {run}{res}')
    print(cmd)
    get_ipython().system(cmd)
    prev = sorted((P.RUNS / run).glob('preview_ep*.png'))
    if prev:
        print(prev[-1]); display(Image(str(prev[-1])))
print(list(CONTROL))

### 7a. Współczynnik uczenia zwykłego U-Netu (po rozłączeniu: `train_control(..., resume=True)`)

In [ ]:
train_control(f'{DAY}_unet_plain_lr1e-4')

In [ ]:
train_control(f'{DAY}_unet_plain_lr1e-3')

### 7b. Szersza sieć `corr` (width 64)

In [ ]:
train_control(f'{DAY}_corr_w64')

## 8. Porównanie
Pary są brane z `results.json` (najnowszy run o danej nazwie, także z 04 i 08); testy kontrolne pokazują się, jeśli były uruchomione. Wszystkie modele od zera: 200 epok.

In [ ]:
import json, pandas as pd
def metrics(rj):
    m = json.load(open(rj))
    return m.get('val_metrics') or m.get('metrics') or m
rows = []
def add(question, label, rj):
    if rj and rj.exists():
        m = metrics(rj)
        rows.append({'pytanie': question, 'model': label, 'run': rj.parent.name, 'psnr': m.get('psnr'),
                     'psnr_full': m.get('psnr_full'), 'ssim': m.get('ssim'), 'wejście (full)': m.get('psnr_full_in')})
    else:
        print('brak', label, rj)
plain_long, corr200 = P.find_run('unet_plain_long', 'results.json'), P.find_run('corr200', 'results.json')
add('A: kamera B', 'zwykły U-Net, A + B', plain_long)
add('A: kamera B', 'zwykły U-Net, tylko A', P.RUNS / RUNS['plain_single'] / 'results.json')
add('A: kamera B', 'corr, A + B', corr200)
add('A: kamera B', 'corr, tylko A', P.RUNS / RUNS['corr_single'] / 'results.json')
add('B: geometria', 'corr (bez geometrii)', corr200)
add('B: geometria', 'korelacja + geometria (od zera)', P.RUNS / RUNS['corrgeom'] / 'results.json')
add('B: geometria', 'RAFT gotowy + geometria v1 (04, bez uczenia)', P.find_run('raft_zeroshot', 'eval.json'))
add('B: geometria', 'RAFT dostrojony + geometria v1 (04, 10 ep.)', P.find_run('raft_geom', 'results.json'))
add('C: kontrola', 'plain, lr 0.0003, width 32', plain_long)
add('C: kontrola', 'corr, lr 0.0003, width 32', corr200)
for run, (mode, lr, width) in CONTROL.items():
    rj = P.RUNS / run / 'results.json'
    if rj.exists():
        add('C: kontrola', f'{mode}, lr {lr:g}, width {width}', rj)
display(pd.DataFrame(rows).set_index(['pytanie', 'model']).round(2))
print('val rs_video_x4k (321 próbek); psnr = obszar widoczny w A, psnr_full = cały obraz (dla RAFT z 04 tylko psnr)')